# v9c: lesion DURING normal continual training, 30% switched off (GPU 0)
Same as `nextphase_v9.ipynb`, but **30%** of each layer's working neurons are switched off instead of 10%.
Results: `analysis/v9/nextphase_d30/`. Start checkpoints: distant 30, 40; related 10, 15. Conditions: nothing / self / look-alike / random.
Seed 0 of everything first. Re-running skips what is done.

In [ ]:
import os, sys, glob, json, time, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
GPU = "0"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
from IPython.display import Image, display
from huggingface_hub import HfApi, hf_hub_download
for attempt in range(5):
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; FILES = api.list_repo_files(HF_REPO); break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)

def get(run, phases):
    for ph in phases:
        if glob.glob(f"runs/{run}/ckpt/p{ph:03d}_*.pt"):
            continue
        name = [f for f in FILES if f.startswith(f"{run}/ckpt/p{ph:03d}_")]
        if name:
            hf_hub_download(HF_REPO, name[0], local_dir="runs")
        else:
            print("missing (not here, not on HF):", run, ph)

def last_phase(run):
    phs = [int(os.path.basename(f)[1:4]) for f in FILES if f.startswith(f"{run}/ckpt/p")]
    phs += [int(os.path.basename(f)[1:4]) for f in glob.glob(f"runs/{run}/ckpt/p*.pt")]
    return max(phs)

def run(langset, *cmd):
    env = {**os.environ, "CML_LANGSET": langset}
    p = subprocess.Popen([sys.executable, "-u", "newtask_v9.py", "--gpu", GPU, "--tasks", "next", "--out", "nextphase_d30", "--dose", "0.30", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        print(line, end="")
    p.wait(); print("exit code", p.returncode)

DIST, REL = "v4b_continual_s0", "v8_indic_continual_s0"
STARTS = [("distant", DIST, 30), ("distant", DIST, 40), ("indic", REL, 10), ("indic", REL, 15)]
for _, r, ph in STARTS:
    get(r, [ph])
print("start models:", [(r, ph) for _, r, ph in STARTS])

In [ ]:
need = [f"data/en-{l}/train.jsonl" for l in ["hi", "zh", "gu"]] + [f"data/flores/devtest.{l}.jsonl" for l in ["en", "hi", "zh", "gu"]]
need += [f"data/flores/dev.{l}.jsonl" for l in ["en", "hi", "zh", "ar", "gu", "pa"]] + [f"data/en-{l}/val.jsonl" for l in ["hi", "zh", "ar", "gu", "pa"]]
miss = [f for f in need if not os.path.exists(f)]
print("all there" if not miss else f"MISSING: {miss}  -> run your download snippet first, then re-run this cell")

In [ ]:
for s in ["0", "1", "2"]:
    for langset, r, ph in STARTS:
        run(langset, "--start", f"{'distant' if langset == 'distant' else 'related'}{ph}={r}:{ph}", "--seeds", s)
    subprocess.run([sys.executable, "newtask_v9.py", "--summarize", "--out", "nextphase_d30"])

In [ ]:
subprocess.run([sys.executable, "newtask_v9.py", "--summarize", "--out", "nextphase_d30"])
for f in ["fig1_learning_curves_loss.png", "fig2_learning_curves_score.png", "fig3_near_to_far.png"]:
    display(Image(f"analysis/v9/nextphase_d30/{f}"))